<a href="https://colab.research.google.com/github/fauzia-annisa/ModShield-Cascaded-LLM-Moderation-Agent/blob/main/ModShield_Cascaded_LLM_Moderation_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 0: Colab setup

In [2]:
!pip -q install -U datasets transformers accelerate scikit-learn
import torch; print(torch.cuda.is_available())

False


Step 1: Load real data

In [3]:
import pandas as pd, numpy as np, time
from datasets import load_dataset

raw = load_dataset("google/civil_comments")
tr = raw["train"].shuffle(seed=42).select(range(60000)).to_pandas()
va = raw["validation"].shuffle(seed=42).select(range(5000)).to_pandas()
te = raw["test"].shuffle(seed=42).select(range(10000)).to_pandas()
for d in (tr, va, te):
    d["y"] = (d["toxicity"] >= 0.5).astype(int)

print(tr.y.mean(), tr.columns.tolist())
ID = [c for c in ["male","female","homosexual_gay_or_lesbian","christian",
      "jewish","muslim","black","white","psychiatric_or_mental_illness"] if c in te.columns]
print(ID)

README.md:   0%|          | 0.00/7.73k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  194MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  187MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97320 [00:00<?, ? examples/s]

0.07825 ['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit', 'y']
[]


Step 2: Metrics helpers

In [4]:
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

def thr_from_val(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    f = 2*pr*rc/(pr+rc+1e-9)
    return th[f[:-1].argmax()]

def f1_at(y, p, t):
    pred = p >= t
    tp = (pred & (y==1)).sum(); fp = (pred & (y==0)).sum(); fn = (~pred & (y==1)).sum()
    return 2*tp/(2*tp+fp+fn+1e-9)

def report(name, col):
    t = thr_from_val(va.y, va[col])
    print(f"{name}: PR-AUC={average_precision_score(te.y, te[col]):.3f} "
          f"ROC-AUC={roc_auc_score(te.y, te[col]):.3f} F1={f1_at(te.y, te[col], t):.3f} thr={t:.2f}")
    return t

Step 3: Baseline (Stage 1)



In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=300000, sublinear_tf=True)
Xtr = vec.fit_transform(tr.text)
lr = LogisticRegression(C=4, max_iter=300, class_weight="balanced").fit(Xtr, tr.y)

va["p_lr"] = lr.predict_proba(vec.transform(va.text))[:,1]
t0 = time.time()
te["p_lr"] = lr.predict_proba(vec.transform(te.text))[:,1]
lat_lr = (time.time()-t0)/len(te)*1000
thr_lr = report("TFIDF+LR", "p_lr"); print("ms/comment", lat_lr)

Step 4: Fine-tune DistilBERT (Stage 2, PyTorch)

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
from scipy.special import softmax

mn = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(mn)

def enc(df):
    ds = Dataset.from_dict({"text": df.text.tolist(), "label": df.y.tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)

dtr, dva, dte = enc(tr.sample(30000, random_state=1)), enc(va), enc(te)
model = AutoModelForSequenceClassification.from_pretrained(mn, num_labels=2)
args = TrainingArguments("bert_out", per_device_train_batch_size=32,
        per_device_eval_batch_size=64, num_train_epochs=2, learning_rate=3e-5,
        fp16=True, save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=dtr,
                  data_collator=DataCollatorWithPadding(tok))
trainer.train()

def predict(ds): return softmax(trainer.predict(ds).predictions, axis=1)[:,1]
va["p_bert"] = predict(dva)
t0 = time.time(); te["p_bert"] = predict(dte)
lat_bert = (time.time()-t0)/len(te)*1000
thr_bert = report("DistilBERT", "p_bert"); print("ms/comment", lat_bert)
trainer.save_model("model"); tok.save_pretrained("model")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Step,Training Loss


Step,Training Loss


Step 5: The LLM agent (Stage 3)

In [ ]:
from transformers import AutoModelForCausalLM
from sklearn.neighbors import NearestNeighbors

lm_name = "Qwen/Qwen2.5-1.5B-Instruct"
ltok = AutoTokenizer.from_pretrained(lm_name, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(lm_name, torch_dtype=torch.float16,
                                           device_map="cuda").eval()
nn = NearestNeighbors(n_neighbors=4, metric="cosine").fit(Xtr)

POLICY = """You are a content-moderation agent for a Q&A platform.
Label A (TOXIC): harassment, insults or personal attacks, threats, hate toward a group or identity, obscene or sexual abuse.
Label B (OK): criticism, disagreement, profanity not aimed at anyone, discussing or quoting offensive content, neutral mention of an identity group.
Answer with a single letter."""

def build(text, ex):
    shots = "\n".join(f"Comment: {t[:200]!r}\nLabel: {'A' if y else 'B'}" for t, y in ex)
    msgs = [{"role":"system","content":POLICY},
            {"role":"user","content":f"Similar past cases:\n{shots}\n\nLabel this comment.\nComment: {text[:600]!r}\nAnswer A or B."}]
    return ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

A = ltok.encode("A", add_special_tokens=False)[0]
B = ltok.encode("B", add_special_tokens=False)[0]

@torch.no_grad()
def llm_score(texts, bs=8):
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i+bs]
        _, idx = nn.kneighbors(vec.transform(chunk))
        prompts = [build(t, [(tr.text.iloc[j], tr.y.iloc[j]) for j in ix])
                   for t, ix in zip(chunk, idx)]
        b = ltok(prompts, return_tensors="pt", padding=True).to("cuda")
        lg = llm(**b).logits[:, -1, :]
        out += torch.softmax(lg[:, [A, B]].float(), -1)[:, 0].cpu().tolist()
    return out

# Only score the uncertain band (this is the cost saving)
for d in (va, te):
    m = (d.p_bert > 0.1) & (d.p_bert < 0.9)
    print(m.sum(), "rows in band")
    t0 = time.time()
    d.loc[m, "p_llm"] = llm_score(d.loc[m, "text"].tolist())
    if d is te: lat_llm = (time.time()-t0)/m.sum()*1000
print("LLM ms/comment", lat_llm)

Step 6: Cascade and the cost vs. accuracy table

In [ ]:
def cascade(df, lo, hi, w=0.5):
    s = df.p_bert.copy()
    m = (df.p_bert > lo) & (df.p_bert < hi) & df.p_llm.notna()
    s[m] = w*df.p_bert[m] + (1-w)*df.p_llm[m]
    return s, m.mean()

rows = []
for lo, hi in [(0.5,0.5),(0.4,0.6),(0.25,0.75),(0.1,0.9)]:
    sv, _ = cascade(va, lo, hi); st, frac = cascade(te, lo, hi)
    t = thr_from_val(va.y, sv)
    rows.append(dict(band=f"{lo}-{hi}", pct_to_llm=round(frac*100,1),
        PR_AUC=round(average_precision_score(te.y, st),3),
        F1=round(f1_at(te.y, st, t),3),
        ms_per_comment=round(lat_bert + frac*lat_llm,2)))
print(pd.DataFrame(rows))

Step 7: Bias audit

In [ ]:
thr = {"p_lr": thr_lr, "p_bert": thr_bert}
rows = []
for c in ID:
    m = (te[c].fillna(0) >= 0.5) & (te.y == 0)
    if m.sum() >= 30:
        rows.append(dict(group=c, n=int(m.sum()),
            FPR_lr=round((te.loc[m,"p_lr"]>thr["p_lr"]).mean(),3),
            FPR_bert=round((te.loc[m,"p_bert"]>thr["p_bert"]).mean(),3)))
base = te.y==0
print("overall FPR bert:", (te.loc[base,"p_bert"]>thr["p_bert"]).mean())
print(pd.DataFrame(rows))

Step 8: Failure analysis

In [ ]:
te["pred"] = (te.p_bert >= thr_bert).astype(int)
print("FALSE POSITIVES"); print(te[(te.pred==1)&(te.y==0)].sort_values("p_bert",ascending=False).text.head(15).to_string())
print("FALSE NEGATIVES"); print(te[(te.pred==0)&(te.y==1)].sort_values("p_bert").text.head(15).to_string())

Step 9: Monitoring (drift)

In [ ]:
def psi(a, b, bins=10):
    edges = np.quantile(a, np.linspace(0,1,bins+1)); edges[0], edges[-1] = -1e-9, 1+1e-9
    pa = np.histogram(a, edges)[0]/len(a)+1e-6; pb = np.histogram(b, edges)[0]/len(b)+1e-6
    return float(np.sum((pa-pb)*np.log(pa/pb)))

print("PSI val->test:", psi(va.p_bert.values, te.p_bert.values))
# simulate shift: attack phase with 3x more toxic content
shift = pd.concat([te[te.y==1], te[te.y==0].sample(len(te[te.y==1])*4, random_state=0)])
print("PSI val->shifted:", psi(va.p_bert.values, shift.p_bert.values))